# VSHORAD - Section 6.1: YOLO Detection Evaluation

**Goal**: Generate all evaluation materials for the YOLO detection models.

## System Architecture

| Tier | Model | imgsz | Target GPU | Notes |
|------|-------|-------|--------------|-------|
| **Strategic** | YOLOv8l | 1280 | A100 40/80GB | Maximum accuracy |
| **Tactical** | YOLOv8m | 960 | L4/T4 ~24GB | Accuracy/speed balance |
| **Embedded** | YOLOv8m TensorRT | 640 | T4/Jetson | Export of Tactical model |

# 1. Setup

In [ ]:
# Install dependencies
!pip install -q ultralytics pandas matplotlib seaborn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import shutil
import json
from datetime import datetime
from ultralytics import YOLO

# Matplotlib config
plt.rcParams['figure.figsize'] = (14, 10)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10
sns.set_style("whitegrid")

print(" Libraries imported successfully")

In [ ]:
# Path config - adjust to your folder structure

import zipfile
import os
from tqdm.notebook import tqdm

BASE_PATH = Path("/content/drive/MyDrive/Inżynierka")

# DATASETY (zipy)
DATASETS_PATH = BASE_PATH / "Datasety Finalne/Datasety_Zbalansowane"
YOLO_ZIP = DATASETS_PATH / "Skydetect_Balanced_Yolo_Dataset.zip"
SWIN_ZIP = DATASETS_PATH / "Skydetect_Balanced_Swin_Dataset.zip"

# Extraction target (local for speed)
DATASET_EXTRACT_PATH = Path("/content/datasets")

# data.yaml - save next to zips
DATASET_YAML = DATASETS_PATH / "data.yaml"

# Models and checkpoints
PATHS = {
  'strategic': {
    'weights': BASE_PATH / "Strategic_components/checkpoints/yolo/strategic_yolov8l_1280/weights/best.pt",
    'results_csv': BASE_PATH / "Strategic_components/checkpoints/yolo/strategic_yolov8l_1280/results.csv",
    'model_name': 'YOLOv8l',
    'imgsz': 1280,
    'tier_name': 'Strategic'
  },
  'tactical': {
    'weights': BASE_PATH / "Medium_components/checkpoints/yolo/medium_yolov8m_960/weights/best.pt",
    'results_csv': BASE_PATH / "Medium_components/checkpoints/yolo/medium_yolov8m_960/results.csv",
    'model_name': 'YOLOv8m',
    'imgsz': 960,
    'tier_name': 'Tactical'
  },
  'embedded': {
    'engine_fp16': BASE_PATH / "Small_components/exports/yolo/yolov8m_640_tensorrt/yolov8m_640_fp16.engine",
    'model_name': 'YOLOv8m TensorRT',
    'imgsz': 640,
    'tier_name': 'Embedded'
  }
}

# OUTPUT FOLDER - REPORTS
REPORTS_BASE = BASE_PATH / "Raporty"

OUTPUT_DIRS = {
  'yolo': REPORTS_BASE / "Yolo",
  'swin': REPORTS_BASE / "Swin",
  'dataset': REPORTS_BASE / "Dataset",
  'latency': REPORTS_BASE / "Latency",
  # Podfoldery dla YOLO
  'tables': REPORTS_BASE / "Yolo/tables",
  'training_curves': REPORTS_BASE / "Yolo/training_curves",
  'validation_strategic': REPORTS_BASE / "Yolo/validation_strategic",
  'validation_tactical': REPORTS_BASE / "Yolo/validation_tactical",
  'comparisons': REPORTS_BASE / "Yolo/comparisons"
}

# Creating folder structure
for dir_path in OUTPUT_DIRS.values():
  dir_path.mkdir(parents=True, exist_ok=True)

print(" Report folder structure created:")
for name, path in OUTPUT_DIRS.items():
  print(f"  {name}: {path}")

# Zip extraction with progress bar
def extract_zip_with_progress(zip_path, extract_to, desc="Extracting"):
  """Extract ZIP with progress bar."""
  with zipfile.ZipFile(zip_path, 'r') as zf:
    members = zf.namelist()
    total_size = sum(info.file_size for info in zf.infolist())

    with tqdm(total=total_size, unit='B', unit_scale=True, desc=desc) as pbar:
      for member in members:
        zf.extract(member, extract_to)
        info = zf.getinfo(member)
        pbar.update(info.file_size)

# Extract YOLO dataset (auto-detect structure)
print(f"\n Extracting datasetu YOLO...")

YOLO_EXTRACT_PATH = DATASET_EXTRACT_PATH / "yolo"
YOLO_DATASET_PATH = None # Will be set after auto-detection

if YOLO_ZIP.exists():
  YOLO_EXTRACT_PATH.mkdir(parents=True, exist_ok=True)

  # Check if already extracted
  existing_images = list(YOLO_EXTRACT_PATH.rglob("images"))

  if not existing_images:
    extract_zip_with_progress(YOLO_ZIP, YOLO_EXTRACT_PATH, desc=" YOLO Dataset")
    print(f"  Extracted do: {YOLO_EXTRACT_PATH}")
  else:
    print(f"  Already extracted: {YOLO_EXTRACT_PATH}")

  # Auto-detect actual dataset path
  for root, dirs, files in os.walk(YOLO_EXTRACT_PATH):
    if 'images' in dirs and 'labels' in dirs:
      YOLO_DATASET_PATH = Path(root)
      break

  if YOLO_DATASET_PATH:
    print(f"  Dataset path: {YOLO_DATASET_PATH}")

    # Show structure
    print(f"\n  Struktura:")
    for item in sorted(YOLO_DATASET_PATH.iterdir()):
      if item.is_dir():
        subcount = len(list(item.iterdir()))
        print(f"   {item.name}/ ({subcount} subdirs)")
  else:
    print(f"  images/labels structure not found in extracted zip!")
else:
  print(f"  Nie znaleziono ZIP: {YOLO_ZIP}")

# Generate data.yaml
print(f"\n Generating data.yaml...")

CLASS_NAMES = [
  'Fighter', 'Helicopter', 'Transport', 'Bomber', 'Special', 'UAV_Fixed',
  'UAV_Rotor', 'Missile', 'Civilian', 'Birds', 'Decoys', 'Unidentified'
]

yaml_content = f"""# VSHORAD Detection Dataset
# Auto-generated data.yaml

path: {YOLO_DATASET_PATH}
train: images/train
val: images/val
test: images/test

nc: {len(CLASS_NAMES)}
names:
"""
for i, name in enumerate(CLASS_NAMES):
  yaml_content += f" {i}: {name}\n"

# Save data.yaml
with open(DATASET_YAML, 'w') as f:
  f.write(yaml_content)

print(f"  Saved: {DATASET_YAML}")
print(f"\n  Contents:")
print("-" * 50)
print(yaml_content)

In [ ]:
# Verify files exist
print("\nVerifying files:")
print("-" * 60)

files_to_check = [
  ("Strategic weights", PATHS['strategic']['weights']),
  ("Strategic results.csv", PATHS['strategic']['results_csv']),
  ("Tactical weights", PATHS['tactical']['weights']),
  ("Tactical results.csv", PATHS['tactical']['results_csv']),
  ("Dataset YAML", DATASET_YAML)
]

all_found = True
for name, path in files_to_check:
  exists = path.exists()
  status = "" if exists else ""
  print(f"{status} {name}: {path}")
  if not exists:
    all_found = False

# Check extracted dataset structure
print("\n Verifying YOLO dataset:")
print("-" * 60)
dataset_dirs = [
  ("Train images", YOLO_DATASET_PATH / "images/train"),
  ("Train labels", YOLO_DATASET_PATH / "labels/train"),
  ("Val images", YOLO_DATASET_PATH / "images/val"),
  ("Val labels", YOLO_DATASET_PATH / "labels/val"),
]

for name, path in dataset_dirs:
  if path.exists():
    count = len(list(path.iterdir()))
    print(f" {name}: {count} files")
  else:
    print(f" {name}: {path}")
    all_found = False

if all_found:
  print("\n All required files found!")
else:
  print("\n Some files not found. Check paths.")

print(f"\n Classes ({len(CLASS_NAMES)}): {', '.join(CLASS_NAMES)}")

In [ ]:
# YOLO classes (12 meta-categories)
CLASS_NAMES = [
  'Fighter', 'Helicopter', 'Transport', 'Bomber', 'Special', 'UAV_Fixed',
  'UAV_Rotor', 'Missile', 'Civilian', 'Birds', 'Decoys', 'Unidentified'
]

print(f" Classes ({len(CLASS_NAMES)}): {', '.join(CLASS_NAMES)}")

# 2. Load Training Data (results.csv)

In [ ]:
def load_training_results(csv_path, tier_name):
  """
  Load results.csv from YOLO training, return DataFrame.
  """
  if not csv_path.exists():
    print(f" Nie znaleziono: {csv_path}")
    return None

  df = pd.read_csv(csv_path)
  # Remove spaces from column names (YOLO sometimes adds them)
  df.columns = df.columns.str.strip()
  df['tier'] = tier_name

  print(f" {tier_name}: Loaded {len(df)} epok")
  return df

# Load training data
df_strategic = load_training_results(PATHS['strategic']['results_csv'], 'Strategic')
df_tactical = load_training_results(PATHS['tactical']['results_csv'], 'Tactical')

In [ ]:
# Show available columns
if df_strategic is not None:
  print("\n Available columns in results.csv:")
  for i, col in enumerate(df_strategic.columns, 1):
    print(f" {i:2}. {col}")

In [ ]:
def extract_final_metrics(df, tier_config):
  """
  Extract final metrics from last row of results.csv.
  """
  if df is None:
    return None

  last_row = df.iloc[-1]

  # Mapowanie nazw kolumn YOLO na nasze
  metrics = {
    'tier': tier_config['tier_name'],
    'model': tier_config['model_name'],
    'imgsz': tier_config['imgsz'],
    'epochs': int(last_row.get('epoch', len(df))),
    'precision': last_row.get('metrics/precision(B)', None),
    'recall': last_row.get('metrics/recall(B)', None),
    'mAP50': last_row.get('metrics/mAP50(B)', None),
    'mAP50_95': last_row.get('metrics/mAP50-95(B)', None)
  }

  return metrics

# Extract final metrics
metrics_strategic = extract_final_metrics(df_strategic, PATHS['strategic'])
metrics_tactical = extract_final_metrics(df_tactical, PATHS['tactical'])

print("\n Final metrics:")
print("-" * 60)

for name, metrics in [('Strategic', metrics_strategic), ('Tactical', metrics_tactical)]:
  if metrics:
    print(f"\n{name} ({metrics['model']}@{metrics['imgsz']}):")
    print(f" Epochs:   {metrics['epochs']}")
    print(f" Precision: {metrics['precision']:.4f}" if metrics['precision'] else " Precision: N/A")
    print(f" Recall:   {metrics['recall']:.4f}" if metrics['recall'] else " Recall:   N/A")
    print(f" mAP@0.5:  {metrics['mAP50']:.4f}" if metrics['mAP50'] else " mAP@0.5:  N/A")
    print(f" mAP@0.5:0.95: {metrics['mAP50_95']:.4f}" if metrics['mAP50_95'] else " mAP@0.5:0.95: N/A")

## 2.1 Training Curves

In [ ]:
def plot_training_curves(df, tier_name, save_path):
  """
  Generate 2x5 training curves plot similar to YOLO results.png.
  """
  if df is None:
    print(f" No data for {tier_name}")
    return

  fig, axes = plt.subplots(2, 5, figsize=(20, 8))
  fig.suptitle(f'Training Curves - {tier_name}', fontsize=14, fontweight='bold')

  # Columns to plot
  plots_config = [
    # Row 1: Train losses + metrics
    ('train/box_loss', 'Box Loss (Train)', 'tab:blue'),
    ('train/cls_loss', 'Cls Loss (Train)', 'tab:orange'),
    ('train/dfl_loss', 'DFL Loss (Train)', 'tab:green'),
    ('metrics/precision(B)', 'Precision', 'tab:purple'),
    ('metrics/recall(B)', 'Recall', 'tab:red'),
    # Row 2: Val losses + mAP
    ('val/box_loss', 'Box Loss (Val)', 'tab:blue'),
    ('val/cls_loss', 'Cls Loss (Val)', 'tab:orange'),
    ('val/dfl_loss', 'DFL Loss (Val)', 'tab:green'),
    ('metrics/mAP50(B)', 'mAP@0.5', 'tab:purple'),
    ('metrics/mAP50-95(B)', 'mAP@0.5:0.95', 'tab:red'),
  ]

  epochs = df['epoch'] if 'epoch' in df.columns else range(1, len(df) + 1)

  for idx, (col, title, color) in enumerate(plots_config):
    row, col_idx = idx // 5, idx % 5
    ax = axes[row, col_idx]

    if col in df.columns:
      ax.plot(epochs, df[col], color=color, linewidth=1.5)
      ax.set_title(title)
      ax.set_xlabel('Epoch')
      ax.grid(True, alpha=0.3)

      # Add final value
      final_val = df[col].iloc[-1]
      ax.annotate(f'{final_val:.4f}', xy=(epochs.iloc[-1] if hasattr(epochs, 'iloc') else epochs[-1], final_val),
            fontsize=8, ha='right', va='bottom')
    else:
      ax.text(0.5, 0.5, f'Column not found:\n{col}', ha='center', va='center', transform=ax.transAxes)
      ax.set_title(title)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  print(f" Saved: {save_path}")

# Generate plots
plot_training_curves(df_strategic, 'Strategic (YOLOv8l@1280)',
           OUTPUT_DIRS['training_curves'] / 'results_strategic.png')

plot_training_curves(df_tactical, 'Tactical (YOLOv8m@960)',
           OUTPUT_DIRS['training_curves'] / 'results_tactical.png')

In [ ]:
def plot_training_comparison_overlay(df_strat, df_tact, save_path):
  """
  Comparison - both models on one plot.
  """
  if df_strat is None or df_tact is None:
    print(" No data for comparison")
    return

  fig, axes = plt.subplots(2, 3, figsize=(15, 10))
  fig.suptitle('Training Curves Comparison: Strategic vs Tactical', fontsize=14, fontweight='bold')

  metrics_to_compare = [
    ('train/box_loss', 'Box Loss (Train)'),
    ('val/box_loss', 'Box Loss (Val)'),
    ('metrics/precision(B)', 'Precision'),
    ('metrics/recall(B)', 'Recall'),
    ('metrics/mAP50(B)', 'mAP@0.5'),
    ('metrics/mAP50-95(B)', 'mAP@0.5:0.95'),
  ]

  epochs_strat = df_strat['epoch'] if 'epoch' in df_strat.columns else range(1, len(df_strat) + 1)
  epochs_tact = df_tact['epoch'] if 'epoch' in df_tact.columns else range(1, len(df_tact) + 1)

  for idx, (col, title) in enumerate(metrics_to_compare):
    row, col_idx = idx // 3, idx % 3
    ax = axes[row, col_idx]

    if col in df_strat.columns:
      ax.plot(epochs_strat, df_strat[col], 'g-', linewidth=2, label='Strategic', alpha=0.8)
    if col in df_tact.columns:
      ax.plot(epochs_tact, df_tact[col], 'b-', linewidth=2, label='Tactical', alpha=0.8)

    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(loc='best')
    ax.grid(True, alpha=0.3)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  print(f" Saved: {save_path}")

plot_training_comparison_overlay(df_strategic, df_tactical,
                 OUTPUT_DIRS['comparisons'] / 'training_curves_comparison.png')

# 3. Model Validation (PR/F1/Confusion Matrix)
**NOTE**: This section runs full validation on the test set.
May take several minutes depending on GPU.

In [ ]:
# Check available GPU
import torch
print(f" CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
  print(f"  GPU: {torch.cuda.get_device_name(0)}")
  print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
def run_validation(weights_path, data_yaml, imgsz, output_dir, tier_name, batch_size=8):
  """
  Run YOLO model validation and save plots.
  Returns results object with metrics.
  """
  print(f"\n{'='*60}")
  print(f" Validation: {tier_name}")
  print(f"{'='*60}")

  if not weights_path.exists():
    print(f" Weights not found: {weights_path}")
    return None

  # Load model
  model = YOLO(str(weights_path))
  print(f" Model loaded: {weights_path.name}")

  # Run validation
  results = model.val(
    data=str(data_yaml),
    imgsz=imgsz,
    batch=batch_size,
    plots=True,     # Generuje PR, F1, confusion matrix
    save_json=True,   # Save results in COCO format
    project=str(output_dir),
    name='val',
    exist_ok=True
  )

  print(f"\n Validation results {tier_name}:")
  print(f"  mAP@0.5:   {results.box.map50:.4f}")
  print(f"  mAP@0.5:0.95: {results.box.map:.4f}")
  print(f"  Precision:  {results.box.mp:.4f}")
  print(f"  Recall:    {results.box.mr:.4f}")

  return results

In [ ]:
# Validate Strategic (may need smaller batch for smaller GPUs)
# Dla A100: batch=16, dla L4/T4: batch=4-8

BATCH_STRATEGIC = 32 # Adjust based on GPU

results_val_strategic = run_validation(
  weights_path=PATHS['strategic']['weights'],
  data_yaml=DATASET_YAML,
  imgsz=PATHS['strategic']['imgsz'],
  output_dir=OUTPUT_DIRS['validation_strategic'],
  tier_name='Strategic',
  batch_size=BATCH_STRATEGIC
)

In [ ]:
# Validate Tactical
BATCH_TACTICAL = 32 # Tactical is smaller, can use larger batch

results_val_tactical = run_validation(
  weights_path=PATHS['tactical']['weights'],
  data_yaml=DATASET_YAML,
  imgsz=PATHS['tactical']['imgsz'],
  output_dir=OUTPUT_DIRS['validation_tactical'],
  tier_name='Tactical',
  batch_size=BATCH_TACTICAL
)

In [ ]:
def extract_per_class_metrics(results, class_names):
  """
  Extract per-class metrics from validation results.
  """
  if results is None:
    return None

  # AP@0.5 per classesa
  ap50_per_class = results.box.ap50

  # Create DataFrame
  metrics_list = []
  for i, class_name in enumerate(class_names):
    if i < len(ap50_per_class):
      metrics_list.append({
        'class': class_name,
        'ap50': ap50_per_class[i]
      })

  # Add mean
  metrics_list.append({
    'class': 'ALL (mean)',
    'ap50': results.box.map50
  })

  return pd.DataFrame(metrics_list)

# Extract per-class metrics
per_class_strategic = extract_per_class_metrics(results_val_strategic, CLASS_NAMES)
per_class_tactical = extract_per_class_metrics(results_val_tactical, CLASS_NAMES)

if per_class_strategic is not None:
  print("\n mAP@0.5 per classesa (Strategic):")
  print(per_class_strategic.to_string(index=False))

if per_class_tactical is not None:
  print("\n mAP@0.5 per classesa (Tactical):")
  print(per_class_tactical.to_string(index=False))

# 4. Generate Tables & Comparison Plots

In [ ]:
# A) Summary metrics table

def create_summary_table(metrics_strat, metrics_tact):
  """
  Build summary metrics table for all tiers.
  """
  data = []

  for metrics in [metrics_strat, metrics_tact]:
    if metrics:
      data.append({
        'Tier': metrics['tier'],
        'Model': metrics['model'],
        'imgsz': metrics['imgsz'],
        'Epochs': metrics['epochs'],
        'Precision': f"{metrics['precision']:.3f}" if metrics['precision'] else '—',
        'Recall': f"{metrics['recall']:.3f}" if metrics['recall'] else '—',
        'mAP@0.5': f"{metrics['mAP50']:.3f}" if metrics['mAP50'] else '—',
        'mAP@0.5:0.95': f"{metrics['mAP50_95']:.3f}" if metrics['mAP50_95'] else '—'
      })

  # Add Embedded (TensorRT export - same metrics as Tactical)
  data.append({
    'Tier': 'Embedded',
    'Model': 'YOLOv8m TRT',
    'imgsz': 640,
    'Epochs': '—',
    'Precision': '—',
    'Recall': '—',
    'mAP@0.5': '—',
    'mAP@0.5:0.95': '—'
  })

  return pd.DataFrame(data)

summary_df = create_summary_table(metrics_strategic, metrics_tactical)
print("\n TRAINING METRICS SUMMARY:")
print("="*80)
print(summary_df.to_string(index=False))

In [ ]:
# Save as CSV
summary_csv_path = OUTPUT_DIRS['tables'] / 'metrics_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)
print(f" Saved: {summary_csv_path}")

In [ ]:
# Generate LaTeX
def dataframe_to_latex(df, caption, label):
  """
  Convert DataFrame to Overleaf-ready LaTeX format.
  """
  latex = df.to_latex(
    index=False,
    escape=False,
    column_format='l' * len(df.columns)
  )

  # Wrap w table environment
  full_latex = f"""\\begin{{table}}[htbp]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
{latex}
\\end{{table}}"""

  return full_latex

latex_summary = dataframe_to_latex(
  summary_df,
  caption='YOLO training metrics comparison for different VSHORAD system tiers',
  label='tab:yolo_metrics_summary'
)

print("\n LATEX (gotowy do wklejenia do Overleaf):")
print("="*80)
print(latex_summary)

# Save to file
latex_path = OUTPUT_DIRS['tables'] / 'metrics_summary.tex'
with open(latex_path, 'w', encoding='utf-8') as f:
  f.write(latex_summary)
print(f"\n Saved: {latex_path}")

In [ ]:
# D) Per-class mAP comparison table

def create_per_class_comparison(df_strat, df_tact, class_names):
  """
  Create per-class mAP comparison table.
  """
  data = []

  for class_name in class_names + ['ALL (mean)']:
    row = {'Class': class_name}

    # Strategic
    if df_strat is not None:
      strat_val = df_strat[df_strat['class'] == class_name]['ap50'].values
      row['Strategic mAP@0.5'] = strat_val[0] if len(strat_val) > 0 else None
    else:
      row['Strategic mAP@0.5'] = None

    # Tactical
    if df_tact is not None:
      tact_val = df_tact[df_tact['class'] == class_name]['ap50'].values
      row['Tactical mAP@0.5'] = tact_val[0] if len(tact_val) > 0 else None
    else:
      row['Tactical mAP@0.5'] = None

    # Delta
    if row['Strategic mAP@0.5'] is not None and row['Tactical mAP@0.5'] is not None:
      row['Δ'] = row['Tactical mAP@0.5'] - row['Strategic mAP@0.5']
    else:
      row['Δ'] = None

    data.append(row)

  df = pd.DataFrame(data)
  return df

per_class_comparison = create_per_class_comparison(per_class_strategic, per_class_tactical, CLASS_NAMES)

print("\n PER-CLASS mAP - COMPARISON:")
print("="*80)

# Format for display
display_df = per_class_comparison.copy()
for col in ['Strategic mAP@0.5', 'Tactical mAP@0.5', 'Δ']:
  display_df[col] = display_df[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else '—')

print(display_df.to_string(index=False))

In [ ]:
# Save per-class table
per_class_csv_path = OUTPUT_DIRS['tables'] / 'per_class_map.csv'
per_class_comparison.to_csv(per_class_csv_path, index=False)
print(f" Saved: {per_class_csv_path}")

# LaTeX
latex_per_class = dataframe_to_latex(
  display_df,
  caption='Per-class mAP@0.5 comparison dla modeli Strategic i Tactical',
  label='tab:yolo_per_class_map'
)

latex_per_class_path = OUTPUT_DIRS['tables'] / 'per_class_map.tex'
with open(latex_per_class_path, 'w', encoding='utf-8') as f:
  f.write(latex_per_class)
print(f" Saved: {latex_per_class_path}")

print("\n LATEX:")
print(latex_per_class)

In [ ]:
# E) Comparison bar chart

def plot_comparison_bar_chart(metrics_strat, metrics_tact, save_path):
  """
  Bar chart comparing Strategic vs Tactical.
  """
  if metrics_strat is None or metrics_tact is None:
    print(" No data for chart")
    return

  fig, axes = plt.subplots(1, 4, figsize=(16, 5))
  fig.suptitle('Metrics Comparison: Strategic vs Tactical', fontsize=14, fontweight='bold')

  metrics_config = [
    ('precision', 'Precision'),
    ('recall', 'Recall'),
    ('mAP50', 'mAP@0.5'),
    ('mAP50_95', 'mAP@0.5:0.95')
  ]

  colors = ['#2ecc71', '#3498db'] # Zielony = Strategic, Niebieski = Tactical

  for idx, (key, title) in enumerate(metrics_config):
    ax = axes[idx]

    val_strat = metrics_strat.get(key, 0) or 0
    val_tact = metrics_tact.get(key, 0) or 0

    bars = ax.bar(['Strategic', 'Tactical'], [val_strat, val_tact], color=colors, edgecolor='black', linewidth=1.2)

    # Add values on bars
    for bar, val in zip(bars, [val_strat, val_tact]):
      ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
          f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  print(f" Saved: {save_path}")

plot_comparison_bar_chart(metrics_strategic, metrics_tactical,
             OUTPUT_DIRS['comparisons'] / 'comparison_bar_chart.png')

In [ ]:
# Per-class mAP comparison chart

def plot_per_class_comparison(df_comparison, save_path):
  """
  Per-class mAP bar chart for both models.
  """
  if df_comparison is None:
    print(" No data")
    return

  # Remove ALL (mean) row from plot
  df_plot = df_comparison[df_comparison['Class'] != 'ALL (mean)'].copy()

  fig, ax = plt.subplots(figsize=(14, 6))

  x = np.arange(len(df_plot))
  width = 0.35

  bars1 = ax.bar(x - width/2, df_plot['Strategic mAP@0.5'].fillna(0), width,
          label='Strategic', color='#2ecc71', edgecolor='black')
  bars2 = ax.bar(x + width/2, df_plot['Tactical mAP@0.5'].fillna(0), width,
          label='Tactical', color='#3498db', edgecolor='black')

  ax.set_xlabel('Klasa')
  ax.set_ylabel('mAP@0.5')
  ax.set_title('Per-Class mAP@0.5 Comparison: Strategic vs Tactical', fontsize=12, fontweight='bold')
  ax.set_xticks(x)
  ax.set_xticklabels(df_plot['Class'], rotation=45, ha='right')
  ax.legend()
  ax.set_ylim(0, 1.1)
  ax.grid(axis='y', alpha=0.3)

  plt.tight_layout()
  plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
  plt.show()
  print(f" Saved: {save_path}")

plot_per_class_comparison(per_class_comparison,
             OUTPUT_DIRS['comparisons'] / 'per_class_map_comparison.png')

# 5. Copy Validation Plots
YOLO `model.val(plots=True)` generates plots in the project folder. This section copies them to our output structure.

In [ ]:
# Check what YOLO actually generated
def list_all_plots(val_dir, tier_name):
  """List all PNG files in validation folder."""
  val_path = Path(val_dir) / 'val'

  print(f"\n Files in {tier_name} ({val_path}):")
  print("-" * 60)

  if not val_path.exists():
    print(f"  Folder does not exist!")
    return []

  all_files = list(val_path.rglob("*.png"))

  if not all_files:
    print(f"  No PNG files found")
  else:
    for f in sorted(all_files):
      rel_path = f.relative_to(val_path)
      print(f"  {rel_path}")

  return all_files

# Check both folders
strategic_plots = list_all_plots(OUTPUT_DIRS['validation_strategic'], 'Strategic')
tactical_plots = list_all_plots(OUTPUT_DIRS['validation_tactical'], 'Tactical')

In [ ]:
def copy_validation_plots(val_output_dir, target_dir, tier_name):
  """
  Copy YOLO validation plots to target folder.
  """
  # YOLO saves plots in 'val' subfolder
  source_dir = Path(val_output_dir) / 'val'

  # Mapping: target name -> possible source names (new and old YOLO versions)
  plot_mapping = {
    'PR_curve.png': ['BoxPR_curve.png', 'PR_curve.png'],
    'P_curve.png': ['BoxP_curve.png', 'P_curve.png'],
    'R_curve.png': ['BoxR_curve.png', 'R_curve.png'],
    'F1_curve.png': ['BoxF1_curve.png', 'F1_curve.png'],
    'confusion_matrix.png': ['confusion_matrix.png'],
    'confusion_matrix_normalized.png': ['confusion_matrix_normalized.png']
  }

  print(f"\n Copying validation plots {tier_name}:")
  print(f"  Source: {source_dir}")
  print(f"  Target:  {target_dir}")

  copied = 0
  for target_name, source_names in plot_mapping.items():
    for src_name in source_names:
      src = source_dir / src_name
      if src.exists():
        dst = Path(target_dir) / target_name
        shutil.copy2(src, dst)
        print(f"  {src_name} → {target_name}")
        copied += 1
        break
    else:
      print(f"  Nie znaleziono: {target_name}")

  print(f"  Copied: {copied}/{len(plot_mapping)}")

# Copy plots
if results_val_strategic is not None:
  copy_validation_plots(OUTPUT_DIRS['validation_strategic'],
             OUTPUT_DIRS['validation_strategic'], 'Strategic')

if results_val_tactical is not None:
  copy_validation_plots(OUTPUT_DIRS['validation_tactical'],
             OUTPUT_DIRS['validation_tactical'], 'Tactical')

# 6. Summary & JSON Export

In [ ]:
# Build summary JSON

def create_summary_json(metrics_strat, metrics_tact, per_class_strat, per_class_tact):
  """
  Build comprehensive JSON with all metrics.
  """
  summary = {
    'generated_at': datetime.now().isoformat(),
    'system': 'VSHORAD',
    'section': '6.1 - YOLO Detection Evaluation',
    'tiers': {
      'strategic': {
        'model': 'YOLOv8l',
        'imgsz': 1280,
        'training_metrics': metrics_strat if metrics_strat else None,
        'per_class_ap50': per_class_strat.to_dict('records') if per_class_strat is not None else None
      },
      'tactical': {
        'model': 'YOLOv8m',
        'imgsz': 960,
        'training_metrics': metrics_tact if metrics_tact else None,
        'per_class_ap50': per_class_tact.to_dict('records') if per_class_tact is not None else None
      },
      'embedded': {
        'model': 'YOLOv8m TensorRT FP16',
        'imgsz': 640,
        'note': 'Export of Tactical model - same detection metrics'
      }
    },
    'class_names': CLASS_NAMES
  }

  return summary

summary_json = create_summary_json(metrics_strategic, metrics_tactical,
                  per_class_strategic, per_class_tactical)

# Save JSON
json_path = OUTPUT_DIRS['yolo'] / 'evaluation_summary.json'
with open(json_path, 'w', encoding='utf-8') as f:
  json.dump(summary_json, f, indent=2, default=str)

print(f" Saved JSON: {json_path}")

In [ ]:
# Summary of generated files

print("\n" + "="*80)
print(" SUMMARY - GENERATED FILES")
print("="*80)

def list_generated_files(base_path):
  """List all generated files."""
  all_files = []
  for root, dirs, files in os.walk(base_path):
    for file in files:
      rel_path = os.path.relpath(os.path.join(root, file), base_path)
      all_files.append(rel_path)
  return sorted(all_files)

generated_files = list_generated_files(OUTPUT_DIRS['yolo'])

print(f"\n Output folder: {OUTPUT_DIRS['yolo']}")
print(f"\n Generated files ({len(generated_files)}):")
print("-" * 60)

current_dir = None
for f in generated_files:
  dir_name = os.path.dirname(f)
  if dir_name != current_dir:
    current_dir = dir_name
    print(f"\n  {dir_name}/" if dir_name else "\n  [root]")
  print(f"   • {os.path.basename(f)}")

In [ ]:
# CHECKLIST

print("\n" + "="*80)
print(" CHECKLIST - SECTION 6.1")
print("="*80)

checklist = [
  ("Summary metrics table (CSV)", (OUTPUT_DIRS['tables'] / 'metrics_summary.csv').exists()),
  ("Summary metrics table (LaTeX)", (OUTPUT_DIRS['tables'] / 'metrics_summary.tex').exists()),
  ("Training curves Strategic", (OUTPUT_DIRS['training_curves'] / 'results_strategic.png').exists()),
  ("Training curves Tactical", (OUTPUT_DIRS['training_curves'] / 'results_tactical.png').exists()),
  ("Training curves comparison", (OUTPUT_DIRS['comparisons'] / 'training_curves_comparison.png').exists()),
  ("PR curve Strategic", (OUTPUT_DIRS['validation_strategic'] / 'val' / 'BoxPR_curve.png').exists() or
              (OUTPUT_DIRS['validation_strategic'] / 'PR_curve.png').exists()),
  ("Confusion matrix Strategic", (OUTPUT_DIRS['validation_strategic'] / 'val' / 'confusion_matrix_normalized.png').exists() or
                  (OUTPUT_DIRS['validation_strategic'] / 'confusion_matrix_normalized.png').exists()),
  ("PR curve Tactical", (OUTPUT_DIRS['validation_tactical'] / 'val' / 'BoxPR_curve.png').exists() or
             (OUTPUT_DIRS['validation_tactical'] / 'PR_curve.png').exists()),
  ("Confusion matrix Tactical", (OUTPUT_DIRS['validation_tactical'] / 'val' / 'confusion_matrix_normalized.png').exists() or
                 (OUTPUT_DIRS['validation_tactical'] / 'confusion_matrix_normalized.png').exists()),
  ("Per-class mAP table (CSV)", (OUTPUT_DIRS['tables'] / 'per_class_map.csv').exists()),
  ("Per-class mAP table (LaTeX)", (OUTPUT_DIRS['tables'] / 'per_class_map.tex').exists()),
  ("Comparison bar chart", (OUTPUT_DIRS['comparisons'] / 'comparison_bar_chart.png').exists()),
  ("Per-class mAP chart", (OUTPUT_DIRS['comparisons'] / 'per_class_map_comparison.png').exists()),
  ("JSON z metrykami", (OUTPUT_DIRS['yolo'] / 'evaluation_summary.json').exists()),
]

for item, status in checklist:
  icon = "" if status else ""
  print(f" {icon} {item}")

completed = sum(1 for _, s in checklist if s)
print(f"\n Completed: {completed}/{len(checklist)}")

## Notes

The Embedded tier uses TensorRT engines built for a specific GPU architecture.
Detection metrics are the same as Tactical (same model, just quantized).